In [64]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from xgboost import XGBClassifier

df = pd.read_csv("../data/processed/steam_features.csv")
print(df.shape)

(20024, 56)


In [65]:
# Separate features and target
X = df.drop(columns=['success_tier'])
y = df['success_tier']

# Split into train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)
print(y_train.value_counts())

Training set: (16019, 55)
Test set: (4005, 55)
success_tier
Low       12273
Medium     3073
High        673
Name: count, dtype: int64


In [66]:
# Model 1 - Logistic Regression (baseline)
lr_model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr_model.fit(X_train, y_train)

y_pred_lr = lr_model.predict(X_test)
print("Logistic Regression Results:")
print(classification_report(y_test, y_pred_lr))

Logistic Regression Results:
              precision    recall  f1-score   support

        High       0.31      0.83      0.45       169
         Low       0.93      0.77      0.84      3068
      Medium       0.36      0.47      0.41       768

    accuracy                           0.72      4005
   macro avg       0.53      0.69      0.57      4005
weighted avg       0.79      0.72      0.74      4005



c:\Users\pansh\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [67]:
# Model 2 - Random Forest
rf_model = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
print("Random Forest Results:")
print(classification_report(y_test, y_pred_rf))

Random Forest Results:
              precision    recall  f1-score   support

        High       0.70      0.52      0.60       169
         Low       0.90      0.95      0.93      3068
      Medium       0.65      0.53      0.58       768

    accuracy                           0.85      4005
   macro avg       0.75      0.67      0.70      4005
weighted avg       0.84      0.85      0.85      4005



In [68]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

xgb_model = XGBClassifier(n_estimators=200, random_state=42, eval_metric='mlogloss')
xgb_model.fit(X_train, y_train_encoded)

y_pred_xgb = xgb_model.predict(X_test)
y_pred_xgb = le.inverse_transform(y_pred_xgb)

print("XGBoost Results:")
print(classification_report(y_test, y_pred_xgb))

XGBoost Results:
              precision    recall  f1-score   support

        High       0.63      0.52      0.57       169
         Low       0.91      0.94      0.92      3068
      Medium       0.62      0.57      0.59       768

    accuracy                           0.85      4005
   macro avg       0.72      0.67      0.69      4005
weighted avg       0.84      0.85      0.84      4005



In [69]:


import pickle

pickle.dump(rf_model, open("../models/rf_model.pkl", "wb"))
pickle.dump(xgb_model, open("../models/xgb_model.pkl", "wb"))
pickle.dump(le, open("../models/label_encoder.pkl", "wb"))

print("Models saved!")

Models saved!


In [70]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2]
}

xgb_tuned = XGBClassifier(random_state=42, eval_metric='mlogloss')

grid_search = GridSearchCV(xgb_tuned, param_grid, cv=3, scoring='f1_weighted', n_jobs=-1)
grid_search.fit(X_train, y_train_encoded)

print("Best parameters:", grid_search.best_params_)
print("Best score:", grid_search.best_score_)

Best parameters: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 200}
Best score: 0.8480086558234241


In [71]:
# Train final tuned XGBoost
xgb_final = XGBClassifier(
    learning_rate=0.1,
    max_depth=3,
    n_estimators=200,
    random_state=42,
    eval_metric='mlogloss'
)
xgb_final.fit(X_train, y_train_encoded)

y_pred_final = xgb_final.predict(X_test)
y_pred_final = le.inverse_transform(y_pred_final)

print("Tuned XGBoost Results:")
print(classification_report(y_test, y_pred_final))

Tuned XGBoost Results:
              precision    recall  f1-score   support

        High       0.73      0.55      0.63       169
         Low       0.90      0.96      0.93      3068
      Medium       0.67      0.55      0.61       768

    accuracy                           0.86      4005
   macro avg       0.77      0.69      0.72      4005
weighted avg       0.85      0.86      0.85      4005



In [72]:
pickle.dump(xgb_final, open("../models/xgb_final.pkl", "wb"))
pickle.dump(le, open("../models/label_encoder.pkl", "wb"))
print("Final model saved!")

Final model saved!
